*single star* **double star** _single underscore_ __double underscore__  

# Heading 1
## Heading 2
### Heading 3

**Bold text**
*Italic text*  
***Bold italic text***  |
~~Strikethrough~~  

`Inline code`

> Blockquote example

---

- Unordered list item 1
- Unordered list item 2
    - Nested item

1. Ordered list item 1
2. Ordered list item 2


### Lets start with the voice recording

In [7]:
import pyaudio
import wave
import numpy as np
import time
import threading
from collections import deque
import json
from datetime import datetime
import os

In [8]:
import torch
from silero_vad import load_silero_vad, get_speech_timestamps

In [9]:
class InterviewVoiceRecorder:
    def __init__(self, 
                 sample_rate=16000,  # 16kHz is standard for VAD
                 pause_threshold=1.2,  # seconds of silence to stop recording
                 min_speech_duration=0.5,  # minimum seconds to consider as speech
                 max_recording_duration=60.0):  # max seconds per answer
        
        self.sample_rate = sample_rate
        self.pause_threshold = pause_threshold
        self.min_speech_duration = min_speech_duration
        self.max_recording_duration = max_recording_duration
        
        # Load VAD model (loads once, runs on CPU)
        print("Loading VAD model...")
        self.vad_model = load_silero_vad()
        
        # Audio settings
        self.chunk_size = 512  # samples per chunk
        self.format = pyaudio.paInt16
        self.channels = 1
        
        # For real-time analysis
        self.audio_buffer = deque()
        self.speech_frames = []
        self.is_recording = False
        self.speech_start_time = None
        self.last_speech_time = None
        
        # Metrics tracking
        self.metrics = {
            'total_speech_duration': 0,
            'pause_durations': [],
            'filler_words': [],
            'background_noise_events': [],
            'speech_segments': []
        }
        
        # Filler word patterns (phonetic detection)
        self.filler_patterns = [
            ('um', 'filler_um'),
            ('uh', 'filler_uh'),
            ('er', 'filler_er'),
            ('ah', 'filler_ah'),
            ('like', 'filler_like'),
            ('you know', 'filler_you_know'),
            ('actually', 'filler_actually'),
            ('basically', 'filler_basically'),
            ('so', 'filler_so'),
            ('well', 'filler_well')
        ]
        
        # Noise patterns (energy thresholds)
        self.noise_threshold = 0.02  # Adjust based on testing
        self.cough_threshold = 0.8   # Sharp energy spike
        
        # Initialize PyAudio
        self.audio = pyaudio.PyAudio()
        
    def get_audio_energy(self, audio_chunk):
        """Calculate RMS energy of audio chunk"""
        audio_int16 = np.frombuffer(audio_chunk, dtype=np.int16)
        if len(audio_int16) == 0:
            return 0.0
        rms = np.sqrt(np.mean(audio_int16.astype(np.float32)**2))
        return rms / 32768.0  # Normalize to 0-1
    
    def detect_filler_words_simple(self, audio_chunk, energy_profile):
        """
        Simplified filler detection based on energy patterns
        Real filler detection would need a separate ASR model
        """
        # For now, we'll detect short energy drops that indicate hesitations
        # Full filler detection will come from Whisper transcription
        pass
    
    def detect_background_noise(self, audio_chunk, energy):
        """Detect coughing, laughing, throat clearing"""
        
        # Check for sharp energy spike (cough/throat clear)
        if len(self.audio_buffer) > 0:
            prev_energy = self.get_audio_energy(self.audio_buffer[-1])
            energy_change = abs(energy - prev_energy)
            
            if energy_change > self.cough_threshold and energy > 0.5:
                event = {
                    'type': 'sharp_noise',
                    'timestamp': time.time(),
                    'energy': energy,
                    'possible_cause': 'cough_or_clear_throat'
                }
                self.metrics['background_noise_events'].append(event)
                return True
        return False
    
    def record_until_silence(self, output_file=None):
        """
        Main recording loop - records until user stops talking
        Returns audio data and comprehensive metrics
        """
        
        print("\n🎤 Listening... (speak your answer)")
        print(f"   Will stop after {self.pause_threshold} seconds of silence")
        print(f"   Maximum recording time: {self.max_recording_duration} seconds\n")
        
        stream = self.audio.open(
            format=self.format,
            channels=self.channels,
            rate=self.sample_rate,
            input=True,
            frames_per_buffer=self.chunk_size
        )
        
        frames = []
        silence_chunks = 0
        silence_threshold_chunks = int(self.pause_threshold * self.sample_rate / self.chunk_size)
        speech_detected = False
        recording_started = False
        start_time = time.time()
        segment_start = None
        
        try:
            while True:
                # Read audio chunk
                audio_chunk = stream.read(self.chunk_size, exception_on_overflow=False)
                frames.append(audio_chunk)
                
                # Calculate energy for this chunk
                energy = self.get_audio_energy(audio_chunk)
                
                # Detect speech using VAD (converts bytes to tensor)
                audio_int16 = np.frombuffer(audio_chunk, dtype=np.int16)
                audio_tensor = torch.from_numpy(audio_int16).float() / 32768.0
                
                # VAD expects a tensor and sample rate
                is_speech = self.vad_model(audio_tensor, self.sample_rate).item()
                
                # Real-time feedback
                if is_speech and not speech_detected:
                    # Speech just started
                    speech_detected = True
                    silence_chunks = 0
                    if not recording_started:
                        recording_started = True
                        segment_start = time.time()
                        print("🔴 Recording started...")
                    else:
                        print("   (speech resumed)")
                
                elif not is_speech and speech_detected:
                    # Speech just ended, start silence counter
                    speech_detected = False
                    silence_chunks = 1
                    print("   (pause detected)")
                    
                    # Record the end of this speech segment
                    if segment_start:
                        segment_duration = time.time() - segment_start
                        self.metrics['speech_segments'].append({
                            'start': segment_start,
                            'duration': segment_duration,
                            'words': None  # Will be filled after transcription
                        })
                        segment_start = None
                
                elif not is_speech and recording_started:
                    # We're in silence, increment counter
                    silence_chunks += 1
                
                # Check for noise events
                if recording_started:
                    self.detect_background_noise(audio_chunk, energy)
                
                # Stop conditions
                elapsed_time = time.time() - start_time
                
                # Stop if silence threshold reached
                if recording_started and silence_chunks >= silence_threshold_chunks:
                    print(f"\n⏸️  Silence detected for {self.pause_threshold}s - stopping recording")
                    break
                
                # Stop if max duration reached
                if elapsed_time >= self.max_recording_duration:
                    print(f"\n⏰ Maximum recording time ({self.max_recording_duration}s) reached")
                    break
                
                # Visual indicator for testing
                if not recording_started:
                    if is_speech:
                        print("🎙️", end="", flush=True)
                    else:
                        print(".", end="", flush=True)
        
        except KeyboardInterrupt:
            print("\n\n⏹️ Recording stopped by user")
        
        finally:
            stream.stop_stream()
            stream.close()
            
        # Calculate metrics
        total_duration = time.time() - start_time
        self.metrics['total_recording_duration'] = total_duration
        self.metrics['total_speech_duration'] = sum(s['duration'] for s in self.metrics['speech_segments'])
        self.metrics['number_of_pauses'] = len(self.metrics['speech_segments']) - 1 if len(self.metrics['speech_segments']) > 0 else 0
        
        # Convert frames to numpy array
        audio_data = b''.join(frames)
        audio_array = np.frombuffer(audio_data, dtype=np.int16)
        
        # Save to file if requested
        if output_file:
            self.save_audio(audio_array, output_file)
            self.save_metrics(output_file.replace('.wav', '_metrics.json'))
        
        print(f"\n📊 Recording complete:")
        print(f"   Total time: {total_duration:.1f}s")
        print(f"   Speech time: {self.metrics['total_speech_duration']:.1f}s")
        print(f"   Pauses detected: {self.metrics['number_of_pauses']}")
        print(f"   Noise events: {len(self.metrics['background_noise_events'])}")
        
        return audio_array, self.metrics
    
    def save_audio(self, audio_array, filename):
        """Save audio to WAV file"""
        with wave.open(filename, 'wb') as wf:
            wf.setnchannels(self.channels)
            wf.setsampwidth(self.audio.get_sample_size(self.format))
            wf.setframerate(self.sample_rate)
            wf.writeframes(audio_array.tobytes())
        print(f"💾 Audio saved to: {filename}")
    
    def save_metrics(self, filename):
        """Save detailed metrics to JSON"""
        # Convert timestamps to strings for JSON
        metrics_copy = self.metrics.copy()
        for segment in metrics_copy['speech_segments']:
            if 'start' in segment:
                segment['start'] = segment['start']
        
        for event in metrics_copy['background_noise_events']:
            if 'timestamp' in event:
                event['timestamp'] = event['timestamp']
        
        with open(filename, 'w') as f:
            json.dump(metrics_copy, f, indent=2)
        print(f"📈 Metrics saved to: {filename}")
    
    def analyze_pauses_for_confidence(self):
        """
        Analyze pause patterns to infer confidence levels
        Long pauses mid-sentence = low confidence
        Frequent short pauses = careful/thoughtful
        """
        if len(self.metrics['pause_durations']) < 2:
            return "No significant pauses detected"
        
        avg_pause = np.mean(self.metrics['pause_durations'])
        max_pause = max(self.metrics['pause_durations'])
        
        if avg_pause > 2.0:
            return "LONG PAUSES - Possible low confidence or deep thinking"
        elif avg_pause > 1.0:
            return "MODERATE PAUSES - Thoughtful responses"
        else:
            return "SHORT PAUSES - Fluent but may be rushed"
    
    def __del__(self):
        self.audio.terminate()

Use examples

In [10]:
if __name__ == "__main__":
    # Initialize the recorder
    recorder = InterviewVoiceRecorder(
        pause_threshold=1.2,  # Stop after 1.2s silence
        min_speech_duration=0.5,
        max_recording_duration=45.0
    )
    
    # Simulate an interview question
    print("\n" + "="*60)
    print("🎯 INTERVIEW SIMULATION")
    print("="*60)
    print("\nInterviewer: Tell me about a time you faced a challenge at work.")
    print("\nPress Enter when ready to answer...")
    input()
    
    # Record the answer
    audio, metrics = recorder.record_until_silence(output_file="interview_answer.wav")
    
    # Print detailed analysis
    print("\n" + "="*60)
    print("📊 DETAILED RESPONSE ANALYSIS")
    print("="*60)
    
    print(f"\n⏱️ TIMING:")
    print(f"   Total recording: {metrics['total_recording_duration']:.1f}s")
    print(f"   Actual speech: {metrics['total_speech_duration']:.1f}s")
    print(f"   Silence/thinking: {metrics['total_recording_duration'] - metrics['total_speech_duration']:.1f}s")
    print(f"   Speech segments: {len(metrics['speech_segments'])}")
    
    print(f"\n🗣️ SPEECH PATTERNS:")
    if metrics['speech_segments']:
        for i, seg in enumerate(metrics['speech_segments']):
            print(f"   Segment {i+1}: {seg['duration']:.1f}s")
    
    print(f"\n🔊 BACKGROUND NOISE:")
    if metrics['background_noise_events']:
        for noise in metrics['background_noise_events']:
            print(f"   {noise['possible_cause']} at {noise['timestamp'] - metrics['total_recording_duration']:.1f}s")
    else:
        print("   None detected")
    
    # Confidence analysis based on pauses
    confidence_score = recorder.analyze_pauses_for_confidence()
    print(f"\n💡 CONFIDENCE INSIGHT:")
    print(f"   {confidence_score}")
    
    print("\n" + "="*60)
    print("✅ Next step: Send audio to Whisper for transcription")
    print("="*60)

Loading VAD model...

🎯 INTERVIEW SIMULATION

Interviewer: Tell me about a time you faced a challenge at work.

Press Enter when ready to answer...

🎤 Listening... (speak your answer)
   Will stop after 1.2 seconds of silence
   Maximum recording time: 45.0 seconds



C:\Users\rusha\AppData\Local\Temp\ipykernel_54756\2480319471.py:132: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  audio_tensor = torch.from_numpy(audio_int16).float() / 32768.0


🔴 Recording started...


⏹️ Recording stopped by user
💾 Audio saved to: interview_answer.wav
📈 Metrics saved to: interview_answer_metrics.json

📊 Recording complete:
   Total time: 44.4s
   Speech time: 0.0s
   Pauses detected: 0
   Noise events: 0

📊 DETAILED RESPONSE ANALYSIS

⏱️ TIMING:
   Total recording: 44.4s
   Actual speech: 0.0s
   Silence/thinking: 44.4s
   Speech segments: 0

🗣️ SPEECH PATTERNS:

🔊 BACKGROUND NOISE:
   None detected

💡 CONFIDENCE INSIGHT:
   No significant pauses detected

✅ Next step: Send audio to Whisper for transcription


In [11]:
class SimpleVoiceRecorder:
    def __init__(self, 
                 silence_threshold=0.01,    # Energy below this = silence
                 silence_duration=1.2,      # Seconds of silence to stop
                 min_recording_duration=1.0, # Minimum seconds to record
                 max_recording_duration=30.0):
        
        self.silence_threshold = silence_threshold
        self.silence_duration = silence_duration
        self.min_recording_duration = min_recording_duration
        self.max_recording_duration = max_recording_duration
        
        # Audio settings
        self.chunk_size = 1024
        self.format = pyaudio.paInt16
        self.channels = 1
        self.sample_rate = 16000
        
        self.audio = pyaudio.PyAudio()
        
    def get_audio_energy(self, audio_chunk):
        """Calculate energy level of audio chunk"""
        audio_int16 = np.frombuffer(audio_chunk, dtype=np.int16)
        if len(audio_int16) == 0:
            return 0.0
        # RMS energy
        rms = np.sqrt(np.mean(audio_int16.astype(np.float32)**2))
        return rms / 32768.0
    
    def is_silent(self, energy):
        """Check if energy level indicates silence"""
        return energy < self.silence_threshold
    
    def record_until_silence(self, output_file=None):
        """
        Records until the user stops talking
        Returns audio data and recording duration
        """
        
        print("\n🎤 Recording... Speak now!")
        print(f"   Will stop after {self.silence_duration} seconds of silence")
        print(f"   Maximum recording: {self.max_recording_duration} seconds")
        print("   (Press Ctrl+C to stop manually if needed)\n")
        
        # Open stream
        stream = self.audio.open(
            format=self.format,
            channels=self.channels,
            rate=self.sample_rate,
            input=True,
            frames_per_buffer=self.chunk_size
        )
        
        frames = []
        silent_chunks = 0
        silence_chunks_required = int(self.silence_duration * self.sample_rate / self.chunk_size)
        
        recording_started = False
        start_time = time.time()
        last_speech_time = None
        
        # Visual indicators
        print("[", end="", flush=True)
        
        try:
            while True:
                # Read audio chunk
                audio_chunk = stream.read(self.chunk_size, exception_on_overflow=False)
                energy = self.get_audio_energy(audio_chunk)
                is_silent = self.is_silent(energy)
                
                # Visual feedback
                if not is_silent:
                    print("#", end="", flush=True)  # Speech
                else:
                    print(".", end="", flush=True)  # Silence
                
                if not recording_started and not is_silent:
                    # First speech detected - start recording
                    recording_started = True
                    frames.append(audio_chunk)
                    silent_chunks = 0
                    last_speech_time = time.time()
                    print("\n🔴 RECORDING...", end=" ", flush=True)
                    
                elif recording_started and not is_silent:
                    # Still recording speech
                    frames.append(audio_chunk)
                    silent_chunks = 0
                    last_speech_time = time.time()
                    
                elif recording_started and is_silent:
                    # Silence while recording - add to buffer
                    frames.append(audio_chunk)
                    silent_chunks += 1
                    
                    # Check if we've had enough silence to stop
                    if silent_chunks >= silence_chunks_required:
                        # Check if we've met minimum recording duration
                        current_duration = time.time() - start_time
                        if current_duration >= self.min_recording_duration:
                            print("\n🔇 Silence detected - stopping recording")
                            break
                        else:
                            # Too short, reset silence counter and keep going
                            print(f"\n⚠️  Recording too short ({current_duration:.1f}s), continuing...")
                            silent_chunks = 0
                
                # Check max duration
                current_duration = time.time() - start_time
                if current_duration >= self.max_recording_duration:
                    print(f"\n⏰ Max recording time ({self.max_recording_duration}s) reached")
                    break
                
                # Small delay to prevent CPU hogging
                time.sleep(0.01)
                
        except KeyboardInterrupt:
            print("\n\n⏹️ Recording stopped manually")
            
        finally:
            stream.stop_stream()
            stream.close()
            print("]", flush=True)
        
        total_duration = time.time() - start_time
        
        # Convert frames to numpy array
        audio_data = b''.join(frames)
        audio_array = np.frombuffer(audio_data, dtype=np.int16)
        
        # Save if filename provided
        if output_file and len(audio_array) > 0:
            self.save_audio(audio_array, output_file)
        
        # Print summary
        print(f"\n📊 Recording Summary:")
        print(f"   Duration: {total_duration:.1f} seconds")
        print(f"   Audio samples: {len(audio_array)}")
        print(f"   File size: {len(audio_data) / 1024:.1f} KB")
        
        return audio_array, total_duration
    
    def save_audio(self, audio_array, filename):
        """Save audio to WAV file"""
        with wave.open(filename, 'wb') as wf:
            wf.setnchannels(self.channels)
            wf.setsampwidth(self.audio.get_sample_size(self.format))
            wf.setframerate(self.sample_rate)
            wf.writeframes(audio_array.tobytes())
        print(f"💾 Audio saved to: {filename}")


# Even simpler version with threading for better responsiveness
class ThreadedVoiceRecorder:
    """
    Uses background thread for audio collection
    More responsive and accurate for silence detection
    """
    
    def __init__(self, silence_threshold=0.01, silence_duration=1.2):
        self.silence_threshold = silence_threshold
        self.silence_duration = silence_duration
        self.sample_rate = 16000
        self.chunk_size = 1024
        
        self.audio = pyaudio.PyAudio()
        self.frames = []
        self.is_recording = False
        self.is_speaking = False
        self.silence_start_time = None
        
    def get_energy(self, chunk):
        audio_int16 = np.frombuffer(chunk, dtype=np.int16)
        if len(audio_int16) == 0:
            return 0.0
        return np.sqrt(np.mean(audio_int16.astype(np.float32)**2)) / 32768.0
    
    def record(self):
        """Main recording method - returns when silence detected"""
        
        stream = self.audio.open(
            format=pyaudio.paInt16,
            channels=1,
            rate=self.sample_rate,
            input=True,
            frames_per_buffer=self.chunk_size
        )
        
        print("\n🎤 Ready! Speak into the microphone...")
        print("   (Recording will stop automatically when you pause)\n")
        
        self.frames = []
        self.is_recording = False
        self.silence_start_time = None
        
        # Show waiting indicator
        print("Listening", end="", flush=True)
        
        try:
            while True:
                chunk = stream.read(self.chunk_size, exception_on_overflow=False)
                energy = self.get_energy(chunk)
                
                # Check if someone is speaking
                if energy > self.silence_threshold:
                    # Speech detected
                    if not self.is_recording:
                        # Start recording
                        self.is_recording = True
                        print("\n🔴 RECORDING STARTED 🔴")
                        print("-" * 40)
                    
                    self.frames.append(chunk)
                    self.silence_start_time = None
                    
                    # Visual feedback for speech
                    print("█", end="", flush=True)
                    
                else:
                    # Silence detected
                    if self.is_recording:
                        # We're recording and hit silence
                        self.frames.append(chunk)  # Keep the silence chunk
                        
                        if self.silence_start_time is None:
                            # Silence just started
                            self.silence_start_time = time.time()
                            print("·", end="", flush=True)
                        else:
                            # Silence continuing
                            silence_duration = time.time() - self.silence_start_time
                            print("·", end="", flush=True)
                            
                            # Check if we've had enough silence
                            if silence_duration >= self.silence_duration:
                                print("\n" + "-" * 40)
                                print("🔇 SILENCE DETECTED - Stopping recording")
                                break
                    else:
                        # Not recording yet, just show we're listening
                        print(".", end="", flush=True)
                        time.sleep(0.05)
                
        except KeyboardInterrupt:
            print("\n\n⏹️ Recording stopped manually")
            
        finally:
            stream.stop_stream()
            stream.close()
        
        # Convert to numpy array
        audio_data = b''.join(self.frames)
        audio_array = np.frombuffer(audio_data, dtype=np.int16)
        
        duration = len(audio_array) / self.sample_rate
        
        print(f"\n📊 Recorded {duration:.1f} seconds of audio")
        
        return audio_array, duration
    
    def save(self, audio_array, filename):
        """Save audio to WAV file"""
        with wave.open(filename, 'wb') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(self.audio.get_sample_size(pyaudio.paInt16))
            wf.setframerate(self.sample_rate)
            wf.writeframes(audio_array.tobytes())
        print(f"💾 Saved to: {filename}")

In [12]:
if __name__ == "__main__":
    print("=" * 60)
    print("🎯 VOICE RECORDER TEST - Interview Simulator")
    print("=" * 60)
    
    # Choose which recorder to use
    print("\nChoose recorder:")
    print("1. Simple Recorder (easier to debug)")
    print("2. Threaded Recorder (better responsiveness)")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    if choice == "2":
        recorder = ThreadedVoiceRecorder(silence_threshold=0.01, silence_duration=1.2)
        
        # Simulate interview question
        print("\n" + "=" * 60)
        print("INTERVIEWER: Describe your ideal work environment.")
        print("=" * 60)
        input("\nPress Enter when ready to answer...")
        
        # Record
        audio_data, duration = recorder.record()
        
        if len(audio_data) > 0:
            # Save recording
            recorder.save(audio_data, "interview_answer.wav")
            
            # Simple analysis
            print(f"\n📈 Quick Analysis:")
            print(f"   Speaking time: {duration:.1f} seconds")
            
            # Rough word count estimate (average 2.5 words per second)
            estimated_words = int(duration * 2.5)
            print(f"   Estimated words: ~{estimated_words}")
            
            if duration < 5:
                print("   ⚠️  Response was quite short - try to elaborate more")
            elif duration > 20:
                print("   ✅ Good length - detailed response")
            else:
                print("   ✅ Appropriate response length")
    else:
        recorder = SimpleVoiceRecorder(
            silence_threshold=0.01,
            silence_duration=1.2,
            min_recording_duration=1.0,
            max_recording_duration=30.0
        )
        
        print("\n" + "=" * 60)
        print("INTERVIEWER: Tell me about a time you showed leadership.")
        print("=" * 60)
        input("\nPress Enter when ready to answer...")
        
        audio_data, duration = recorder.record_until_silence(output_file="interview_answer.wav")
    
    print("\n✅ Recording test complete!")
    print("Next step: Integrate with Whisper for transcription")

🎯 VOICE RECORDER TEST - Interview Simulator

Choose recorder:
1. Simple Recorder (easier to debug)
2. Threaded Recorder (better responsiveness)

INTERVIEWER: Tell me about a time you showed leadership.

🎤 Recording... Speak now!
   Will stop after 1.2 seconds of silence
   Maximum recording: 30.0 seconds
   (Press Ctrl+C to stop manually if needed)

[...........................#
🔴 RECORDING... #..###...........####.....#####..................
🔇 Silence detected - stopping recording
]
💾 Audio saved to: interview_answer.wav

📊 Recording Summary:
   Duration: 5.0 seconds
   Audio samples: 51200
   File size: 100.0 KB

✅ Recording test complete!
Next step: Integrate with Whisper for transcription
